In [45]:
# Importation des modules nécessaires
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

# 1. Initialisation de la session Spark (on lui donne un peu de mémoire car le dataset est gros)
spark = SparkSession.builder \
    .appName("Yelp_eALS_Data_Cleaning") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("Session Spark initialisée avec succès !")

# 2. Chargement des données brutes
# /!\ REMPLACE cette variable par le chemin correct vers ton fichier review
file_path = "/content/drive/.shortcut-targets-by-id/1_GN22rAl0NFIidfddFsX0Smnr9phjqN6/data/yelp_academic_dataset_review.json"

print("Chargement du fichier JSON en cours (cela peut prendre quelques instants)...")

# 3. Extraction et création du feedback implicite
# On lit le JSON, on garde 2 colonnes, et on ajoute la colonne 'interaction' valant 1.0
df_implicit = spark.read.json(file_path) \
    .select("user_id", "business_id") \
    .withColumn("interaction", lit(1.0))

# 4. Affichage des résultats
df_implicit.show(5)
print(f"Nombre total de lignes avant filtrage : {df_implicit.count()}")

Session Spark initialisée avec succès !
Chargement du fichier JSON en cours (cela peut prendre quelques instants)...
+--------------------+--------------------+-----------+
|             user_id|         business_id|interaction|
+--------------------+--------------------+-----------+
|mh_-eMZ6K5RLWhZyI...|XQfwVwDr-v0ZS3_Cb...|        1.0|
|OyoGAe7OKpv6SyGZT...|7ATYjTIgM3jUlt4UM...|        1.0|
|8g_iMtfSiwikVnbP2...|YjUWPpI6HXG530lwP...|        1.0|
|_7bHUi9Uuf5__HHc_...|kxX2SOes4o-D3ZQBk...|        1.0|
|bcjbaE6dDog4jkNY9...|e4Vwtrqf-wpJfwesg...|        1.0|
+--------------------+--------------------+-----------+
only showing top 5 rows
Nombre total de lignes avant filtrage : 6990280


In [46]:
from pyspark.ml.feature import StringIndexer

print("1. Comptage et filtrage (>= 10 interactions)...")

# On compte les interactions par utilisateur et on garde ceux qui en ont >= 10
valid_users = df_implicit.groupBy("user_id").count().filter("count >= 10")

# On compte les interactions par commerce et on garde ceux qui en ont >= 10
valid_businesses = df_implicit.groupBy("business_id").count().filter("count >= 10")

# On filtre notre dataframe principal en faisant une jointure (inner join)
# Cela ne gardera que les lignes où l'utilisateur ET le commerce ont au moins 10 interactions
df_filtered = df_implicit.join(valid_users, "user_id", "inner") \
                         .drop("count") \
                         .join(valid_businesses, "business_id", "inner") \
                         .drop("count")

print("2. Conversion des IDs texte en identifiants numériques (Indexation)...")

# Indexation des utilisateurs (crée une nouvelle colonne 'user_idx')
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_idx")
df_indexed = user_indexer.fit(df_filtered).transform(df_filtered)

# Indexation des commerces (crée une nouvelle colonne 'item_idx')
item_indexer = StringIndexer(inputCol="business_id", outputCol="item_idx")
df_final = item_indexer.fit(df_indexed).transform(df_indexed)

# Sélection des colonnes finales propres pour notre algorithme eALS
df_final = df_final.select("user_idx", "item_idx", "interaction")

print("\n--- RÉSULTAT DU NETTOYAGE ---")
df_final.show(5)
print(f"Nombre total de lignes APRÈS filtrage : {df_final.count()}")

1. Comptage et filtrage (>= 10 interactions)...
2. Conversion des IDs texte en identifiants numériques (Indexation)...

--- RÉSULTAT DU NETTOYAGE ---
+--------+--------+-----------+
|user_idx|item_idx|interaction|
+--------+--------+-----------+
| 40006.0| 75234.0|        1.0|
|  6761.0| 75234.0|        1.0|
|  1750.0| 75234.0|        1.0|
| 81700.0| 75234.0|        1.0|
| 74045.0| 75234.0|        1.0|
+--------+--------+-----------+
only showing top 5 rows
Nombre total de lignes APRÈS filtrage : 3162313


In [49]:
print("--- RÉDUCTION INTELLIGENTE DU DATASET ---")

# 1. On trouve les 10 000 utilisateurs les plus actifs
top_users = df_final.groupBy("user_idx").count().orderBy("count", ascending=False).limit(2000).select("user_idx")

# 2. On trouve les 10 000 articles les plus populaires
top_items = df_final.groupBy("item_idx").count().orderBy("count", ascending=False).limit(2000).select("item_idx")

# 3. On filtre notre dataset pour ne garder que les intéractions entre ce groupe d'élite
df_final = df_final.join(top_users, "user_idx", "inner") \
                   .join(top_items, "item_idx", "inner")

# On met en cache et on force le calcul
df_final.cache()
nb_lignes = df_final.count()

print(f"Nouveau dataset dense prêt ! Il contient {nb_lignes} interactions de haute qualité.")
df_final.show(5)

--- RÉDUCTION INTELLIGENTE DU DATASET ---
Nouveau dataset dense prêt ! Il contient 108492 interactions de haute qualité.
+--------+--------+-----------+
|item_idx|user_idx|interaction|
+--------+--------+-----------+
|   681.0|   635.0|        1.0|
|   681.0|  1249.0|        1.0|
|   681.0|  1484.0|        1.0|
|   681.0|  1058.0|        1.0|
|   681.0|  1347.0|        1.0|
+--------+--------+-----------+
only showing top 5 rows


In [50]:
print("--- SÉPARATION TRAIN / TEST ---")

# On divise nos données d'élite : 80% pour apprendre, 20% pour tester
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)

# On met en cache pour accélérer les accès futurs
train_df.cache()
test_df.cache()

# On force le calcul pour voir les chiffres
print(f"Lignes pour l'entraînement (Train) : {train_df.count()}")
print(f"Lignes pour l'évaluation (Test) : {test_df.count()}")

--- SÉPARATION TRAIN / TEST ---
Lignes pour l'entraînement (Train) : 86895
Lignes pour l'évaluation (Test) : 21597


In [51]:
import pyspark.sql.functions as F

print("1. Calcul des poids de popularité (c_i) 100% Distribué...")
c0 = 512.0
alpha = 0.4

# On compte le total directement en Spark
total_interactions = train_df.count()

# On calcule les fréquences et puissances via des opérations de colonnes Spark
item_freq = train_df.groupBy("item_idx").count() \
    .withColumn("f_i", F.col("count") / total_interactions) \
    .withColumn("f_i_alpha", F.pow(F.col("f_i"), alpha))

# On fait la somme distribuée d'un coup
sum_f_alpha = item_freq.select(F.sum("f_i_alpha")).collect()[0][0]

# On calcule le c_i final pour chaque item
item_weights = item_freq.withColumn("c_i", F.lit(c0) * (F.col("f_i_alpha") / F.lit(sum_f_alpha)))

# On ne fait le .collect() qu'à la TOUTE FIN pour créer le dictionnaire
c_dict = {row['item_idx']: row['c_i'] for row in item_weights.select("item_idx", "c_i").collect()}


print("2. Création des historiques avec DataFrames (Sans groupByKey)...")
# Au lieu de groupByKey sur des RDD, on utilise collect_list sur des DataFrames.
# Catalyst va optimiser le shuffle sous le capot beaucoup mieux que groupByKey.

# Pour les utilisateurs
user_history_df = train_df.groupBy("user_idx") \
    .agg(F.collect_list(F.array("item_idx", "interaction")).alias("items"))

# Pour les items
item_history_df = train_df.groupBy("item_idx") \
    .agg(F.collect_list(F.array("user_idx", "interaction")).alias("users"))

# On repasse en RDD uniquement à la fin car ta boucle eALS (Algorithme 1)
# en aura besoin pour itérer facilement en Python local.
user_history_rdd = user_history_df.rdd.map(lambda row: (int(row.user_idx), [(int(x[0]), float(x[1])) for x in row.items])).cache()
item_history_rdd = item_history_df.rdd.map(lambda row: (int(row.item_idx), [(int(x[0]), float(x[1])) for x in row.users])).cache()

print("3. Mise en cache forcée (Action Spark)...")
print(f"Total utilisateurs groupés (Train) : {user_history_rdd.count()}")
print(f"Total articles groupés (Train) : {item_history_rdd.count()}")

1. Calcul des poids de popularité (c_i) 100% Distribué...
2. Création des historiques avec DataFrames (Sans groupByKey)...
3. Mise en cache forcée (Action Spark)...
Total utilisateurs groupés (Train) : 1992
Total articles groupés (Train) : 2000


In [55]:
import time

K = 16          # On commence petit (16) pour tester la vitesse sur Colab
lambda_reg = 0.01
num_iterations = 10 # 1 seule itération pour le test

print("Initialisation des matrices P et Q...")
P = {u: np.random.normal(0, 0.1, K) for u in users_list}
Q = {i: np.random.normal(0, 0.1, K) for i in items_list}

c_bc = spark.sparkContext.broadcast(c_dict)
# --- FONCTION MAPPER 1 : USERS (CORRIGÉE) ---
def update_user_partition(iterator, Q_bc, Sq_bc, P_bc, c_bc, K, l2_reg):
    Q_local, Sq_local, P_local, c_local = Q_bc.value, Sq_bc.value, P_bc.value, c_bc.value
    updated_users = []

    for user_id, interactions in iterator:
        # FIX : On utilise .get() avec une initialisation de secours au lieu de [user_id]
        p_u = P_local.get(user_id, np.random.normal(0, 0.1, K)).copy()

        for f in range(K):
            numerator = 0.0
            denominator = Sq_local[f, f] + l2_reg

            sum_k_neq_f = np.dot(p_u, Sq_local[:, f]) - (p_u[f] * Sq_local[f, f])
            numerator -= sum_k_neq_f

            for item_id, r_ui in interactions:
                # FIX : On sécurise aussi la recherche de l'item
                q_i = Q_local.get(item_id, np.random.normal(0, 0.1, K))
                c_i = c_local.get(item_id, 0.0)
                w_ui = 1.0

                r_hat_f = np.dot(p_u, q_i) - (p_u[f] * q_i[f])
                numerator += (w_ui * r_ui - (w_ui - c_i) * r_hat_f) * q_i[f]
                denominator += (w_ui - c_i) * (q_i[f] ** 2)

            p_u[f] = numerator / denominator

        updated_users.append((user_id, p_u))
    return updated_users


# --- FONCTION MAPPER 2 : ITEMS (CORRIGÉE) ---
def update_item_partition(iterator, P_bc, Sp_bc, Q_bc, c_bc, K, l2_reg):
    P_local, Sp_local, Q_local, c_local = P_bc.value, Sp_bc.value, Q_bc.value, c_bc.value
    updated_items = []

    for item_id, interactions in iterator:
        # FIX : Sécurisation
        q_i = Q_local.get(item_id, np.random.normal(0, 0.1, K)).copy()
        c_i = c_local.get(item_id, 0.0)

        for f in range(K):
            numerator = 0.0
            denominator = c_i * Sp_local[f, f] + l2_reg

            sum_k_neq_f = np.dot(q_i, Sp_local[:, f]) - (q_i[f] * Sp_local[f, f])
            numerator -= c_i * sum_k_neq_f

            for user_id, r_ui in interactions:
                # FIX : Sécurisation
                p_u = P_local.get(user_id, np.random.normal(0, 0.1, K))
                w_ui = 1.0

                r_hat_f = np.dot(p_u, q_i) - (p_u[f] * q_i[f])
                numerator += (w_ui * r_ui - (w_ui - c_i) * r_hat_f) * p_u[f]
                denominator += (w_ui - c_i) * (p_u[f] ** 2)

            q_i[f] = numerator / denominator

        updated_items.append((item_id, q_i))
    return updated_items

Initialisation des matrices P et Q...


In [ ]:
import time
import math
import numpy as np

# 1. PRÉPARATION DU TEST SET (si ce n'est pas déjà fait)
# On convertit le dataframe de test en RDD format (user, item, rating)
test_rdd = test_df.select("user_idx", "item_idx", "interaction") \
    .rdd.map(lambda row: (int(row.user_idx), int(row.item_idx), float(row.interaction))).cache()

# --- FONCTION MAPPER RMSE ---
def calculate_rmse_partition(iterator, P_bc, Q_bc):
    P_local, Q_local = P_bc.value, Q_bc.value
    error_sum = 0.0
    count = 0
    for u, i, r_ui in iterator:
        # On vérifie que l'utilisateur et l'item existent dans nos matrices (Cold Start)
        if u in P_local and i in Q_local:
            pred = np.dot(P_local[u], Q_local[i])
            error_sum += (r_ui - pred) ** 2
            count += 1
    yield (error_sum, count)

# ==========================================
# BOUCLE PRINCIPALE E-ALS AVEC RMSE
# ==========================================

print(f"Lancement de l'entraînement eALS sur {num_iterations} itérations...")

for it in range(num_iterations):
    start_time = time.time()

    # ---------------------------------------------------
    # PHASE 1 : MISE À JOUR DES UTILISATEURS (P)
    # ---------------------------------------------------
    # Optimisation NumPy : Vectorisation de Sq au lieu de la boucle Python
    Q_matrix = np.array([Q.get(i, np.zeros(K)) for i in items_list])
    C_vector = np.array([c_dict.get(i, 0.0) for i in items_list])
    Q_weighted = Q_matrix * np.sqrt(C_vector)[:, np.newaxis]
    Sq = Q_weighted.T @ Q_weighted # Produit matriciel ultra-rapide

    Q_bc = spark.sparkContext.broadcast(Q)
    Sq_bc = spark.sparkContext.broadcast(Sq)
    P_bc = spark.sparkContext.broadcast(P)

    updated_P_rdd = user_history_rdd.mapPartitions(
        lambda partition: update_user_partition(partition, Q_bc, Sq_bc, P_bc, c_bc, K, lambda_reg)
    )
    P.update(updated_P_rdd.collectAsMap())

    Q_bc.unpersist(); Sq_bc.unpersist(); P_bc.unpersist()

    # ---------------------------------------------------
    # PHASE 2 : MISE À JOUR DES ITEMS (Q)
    # ---------------------------------------------------
    # Optimisation NumPy : Vectorisation de Sp
    P_matrix = np.array(list(P.values()))
    Sp = P_matrix.T @ P_matrix

    P_bc = spark.sparkContext.broadcast(P)
    Sp_bc = spark.sparkContext.broadcast(Sp)
    Q_bc = spark.sparkContext.broadcast(Q)

    updated_Q_rdd = item_history_rdd.mapPartitions(
        lambda partition: update_item_partition(partition, P_bc, Sp_bc, Q_bc, c_bc, K, lambda_reg)
    )
    Q.update(updated_Q_rdd.collectAsMap())

    # On garde P_bc et Q_bc en mémoire pour le calcul du RMSE juste après
    Sp_bc.unpersist()

    # ---------------------------------------------------
    # PHASE 3 : CALCUL DU RMSE SUR LE TEST SET
    # ---------------------------------------------------
    rmse_results = test_rdd.mapPartitions(
        lambda partition: calculate_rmse_partition(partition, P_bc, Q_bc)
    ).collect()

    P_bc.unpersist(); Q_bc.unpersist()

    # Agrégation des résultats du RMSE
    total_squared_error = sum([res[0] for res in rmse_results])
    total_predictions = sum([res[1] for res in rmse_results])

    rmse_value = math.sqrt(total_squared_error / total_predictions) if total_predictions > 0 else 0.0

    # ---------------------------------------------------
    # LOGS
    # ---------------------------------------------------
    iter_time = time.time() - start_time
    print(f"Itération {it + 1}/{num_iterations} | Temps: {round(iter_time, 2)}s | Test RMSE: {round(rmse_value, 4)}")

print("Entraînement eALS terminé avec succès !")

Lancement de l'entraînement eALS sur 1 itérations...


In [38]:
import math

print("ÉVALUATION DU MODÈLE SUR LE TEST SET")

# On récupère les données de test localement (comme c'est un échantillon, ça passe en RAM)
test_data = test_df.collect()

squared_error = 0.0
valid_predictions = 0

for row in test_data:
    u = int(row['user_idx'])
    i = int(row['item_idx'])
    actual_rating = row['interaction'] # Vaut 1.0

    # On vérifie qu'on a bien appris un profil pour cet utilisateur et cet article
    # (Certains peuvent n'apparaître QUE dans le test set, c'est le problème du "Cold Start")
    if u in P and i in Q:
        # La prédiction est le produit scalaire des deux vecteurs
        predicted_rating = np.dot(P[u], Q[i])

        # On calcule l'erreur
        squared_error += (actual_rating - predicted_rating) ** 2
        valid_predictions += 1

if valid_predictions > 0:
    rmse = math.sqrt(squared_error / valid_predictions)
    print(f"Prédictions valides réalisées : {valid_predictions} / {len(test_data)}")
    print(f"Score RMSE sur le Test Set : {round(rmse, 4)}")
else:
    print("Aucune prédiction n'a pu être faite (Cold Start total).")

ÉVALUATION DU MODÈLE SUR LE TEST SET
Prédictions valides réalisées : 8732 / 8734
Score RMSE sur le Test Set : 0.8365
